# Feature Engineering

Cargamos el dataset preprocesado para enriquecerlo

In [36]:
import pandas as pd

# Carga del dataset preprocesado
parquet_path = '../data/rfqs_preprocessed.parquet'
model_data = pd.read_parquet(parquet_path)

model_data['start_date'] = pd.to_datetime(model_data['start_date'])
model_data['end_date'] = pd.to_datetime(model_data['end_date'])

train_df = model_data.copy()

train_df.shape

(13796, 26)

Añadimos variables derivadas

In [ ]:
# Feature engineering
train_df['nominal_maturity_months'] = (                                   # Duración nominal en meses
    ((train_df['end_date'] - train_df['start_date']).dt.days / 30.4375)
    .clip(lower=0)
    .apply(lambda x: int(x) if float(x).is_integer() else int(x) + 1)
)

train_df['barrier_spread'] = (                                            # Diferencia entre el porcentaje de barrera de autocall y el de protección
    train_df['autocall_barrier_pct'] - train_df['protection_barrier_pct']
)

train_df['vol_spread'] = (                                                # Diferencia entre la volatilidad implícita cotizada y la volatilidad histórica
    train_df['quoted_implied_vol'] - train_df['vol_63d_mean']
)

train_df['no_call_ratio'] = (                                             # Proporción de meses sin autocall respecto a la duración nominal
    train_df['no_call_period_months'] / train_df['nominal_maturity_months'].replace(0, pd.NA)
)

train_df.shape

(13796, 30)

Transformamos las variables categóricas mediante One-Hot Encoding y guardamos para entrenamiento

In [38]:
train_df.select_dtypes(include='object').nunique().sort_values(ascending=False)

rfq_id                   13796
reference_date            3068
underlyings               1613
trader_id                   39
observation_frequency       18
counterparty                 8
product_type                 6
basket_type                  2
dtype: int64

basket_type solo tiene 2 valores distintos (lo tendremos en cuenta para el One-Hot)

In [39]:
train_df.observation_frequency.value_counts()

observation_frequency
1M            3406
3M            2846
6M            2034
2M            1085
1Y             889
mensual        402
1D             392
M              377
Monthly        363
1 month        355
Q              327
3 months       321
Quarterly      321
trimestral     311
12M             97
Annual          93
Y               92
anual           85
Name: count, dtype: int64

Vamos a unificar ya que hay valores que significan lo mismo, como Y, annual o 1Y

In [40]:
freq_map = {
    # Diaria
    "1D": "daily",

    # Mensual
    "1M": "monthly",
    "M": "monthly",
    "Monthly": "monthly",
    "1 month": "monthly",
    "mensual": "monthly",

    # Bimestral
    "2M": "bimonthly",

    # Trimestral
    "3M": "quarterly",
    "Q": "quarterly",
    "3 months": "quarterly",
    "Quarterly": "quarterly",
    "trimestral": "quarterly",

    # Semestral
    "6M": "semiannual",

    # Anual
    "1Y": "annual",
    "12M": "annual",
    "Annual": "annual",
    "Y": "annual",
    "anual": "annual",
}

train_df["observation_frequency"] = train_df["observation_frequency"].replace(freq_map)

# Comprobación
train_df["observation_frequency"].value_counts()

observation_frequency
monthly       4903
quarterly     4126
semiannual    2034
annual        1256
bimonthly     1085
daily          392
Name: count, dtype: int64

In [41]:
# Eliminamos la columna "trader_id" ya que no es relevante el trader individual para predecir la duración de los autocallables
train_df.drop(columns=['trader_id'], inplace=True)

# Codificación de categóricas (One-Hot)
categorical_cols = [
    'product_type',
    'observation_frequency',
    'counterparty',
]

binary_cols = [
    'basket_type',   # Como tiene solo 2 valores evitamos la columna redundante
]

train_df_encoded = pd.get_dummies(
    train_df,
    columns=[c for c in categorical_cols if c in train_df.columns],
    drop_first=False,
    dtype='int8'
)

train_df_encoded = pd.get_dummies(
    train_df_encoded,
    columns=[c for c in binary_cols if c in train_df_encoded.columns],
    drop_first=True,
    dtype='int8'
)

# Guardamos para notebook de entrenamiento
train_df_encoded.to_parquet("../data/processed_features.parquet", index=False)

print(f"Train encoded guardado en: ../data/processed_features.parquet")
print(f"Shape train_df_encoded: {train_df_encoded.shape}")

Train encoded guardado en: ../data/processed_features.parquet
Shape train_df_encoded: (13796, 46)
